# CS-781 Capstone, Phase 5: Legal Statute Identification (LSI) with the Phase-1 Mamba-2 / attention hybrid (111M)

One notebook, four presets (three main experiments + one optional extra), each producing its own `submission_<EXP>.zip`. Set `EXPERIMENT` in section 1, run all cells, repeat for the next preset.

**Task:** multi-label classification. For every case, predict the set of applicable statute IDs. Metric: macro-F1 over statutes (so rare statutes matter as much as common ones); secondary: micro-F1 and subset accuracy.

**Representation of long cases.** Text is normalised exactly like Phase 1, tokenised lazily, cut into 1024-token chunks (aligned from the document end) and at most `MAX_CHUNKS` chunks are kept (`CHUNK_POLICY`: `uniform` by default because statute citations can appear anywhere; also `first`, `last`, `head_tail`). The backbone encodes every chunk; a chunk vector is the mean of its token states plus the last token's state plus a relative-position embedding.

**Label-wise attention head (`AGG="label_attn"`).** Every statute has its own attention query over the case's chunks, so each statute can look at the section where it is actually discussed, then its own linear scorer produces one logit per statute. Ablations: `attn` (one shared attention), `max` (chunk-level logits, max over chunks), `mean`.

**Multi-label machinery.**
* **Loss** (`LOSS`): `bce`, `bce_pw` (positive-weighted, default), `focal`, `asl` (asymmetric). The output bias starts at each statute's log-odds prior so rare statutes do not begin at 50%.
* **Rare statutes:** training cases are sampled with weight proportional to the rarity of their rarest statute (`SAMPLING="rare_boost"`); the notebook reports F1 by training-frequency bucket.
* **Thresholds:** a global threshold or per-label thresholds shrunk towards the global one (`THRESHOLD_MODE="auto"` picks by a 2-fold estimate on dev, not by the score the thresholds were tuned on). Every case always receives at least one statute (`MIN_PRED=1`).
* **Label co-occurrence:** `LABEL_REFINE=True` adds a low-rank statute-interaction layer (ablation); the analysis reports whether predicted statute pairs occur together in training.
* Predictions contain only valid statute IDs from the train+dev label space, as unique Python integers.

| Preset | Technique | Trainable |
|---|---|---|
| `M1_adapter` | Bottleneck adapters (one per layer) | adapters + head |
| `M2_partial` | Partial fine-tuning (top 5 of 20 layers) | top layers + head |
| `M3_kd` | Knowledge distillation into a half-depth student | whole student |
| `M4_lorma` | (extra) **LORMA** multiplicative low-rank | LORMA matrices + head |

**About LORMA (please read).** The assignment cites LORMA only as reference [12] and does not define it. The implementation is my reconstruction from memory of *multiplicative* low-rank adaptation: `W' = (I + s·B·A)·W0` on the layer output, or `W0·(I + s·B·A)` on the layer input (`side="auto"` picks the cheaper side), with `B` zero-initialised. I could not check the paper, so **compare `LoRMALinear` with reference [12] and edit it if the formulation differs.**

**About QLoRA.** Implemented from scratch: each frozen linear weight is quantised blockwise (64 weights per block, absmax scaling) to 4-bit NF4 codes, stored packed two codes per byte, and de-quantised on the fly in the forward pass; LoRA matrices are trained on top. The 16 NF4 code values are typed from memory of the QLoRA paper [9]: check them against it. The notebook prints the quantisation error and the memory used by the frozen weights.

**Not implemented:** AdaLoRA (also on the Phase-5 list). Prefix tuning exists only in the encoder-decoder notebook.

**Knowledge distillation** uses the teacher's per-statute logits (soft targets with temperature), so any run of any Phase-5 notebook can teach any student, whatever the tokenizer or architecture. Non-KD runs save `teacher_<EXP>.pkl`; the student trains on the training cases it contains.

## 1. Configuration

In [1]:
import os, sys, json, re, math, time, random, zipfile, heapq, shutil, subprocess, hashlib, pickle, glob
from collections import Counter, defaultdict
from dataclasses import dataclass
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint

DEBUG = False                 # True = tiny synthetic smoke test
SEED = 42
EXPERIMENT = "M2_partial"

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
# ---- Phase-1 assets: Mamba-hybrid checkpoint and ITS OWN bpe_merges.json (137M-token-corpus tokenizer) ----
CKPT_PATH = "/kaggle/input/models/ashishkrjha227/mamba-implementation/pytorch/default/1/best.pt"
TOK_PATH  = "/kaggle/input/models/ashishkrjha227/mamba-implementation/pytorch/default/1/bpe_merges.json"
OUTPUT_DIR = "/kaggle/working/phase5_lsi"
EOS_PLACEHOLDER = "<|endoftext|>"
TEACHER_PKL = "/kaggle/working/phase5_lsi/M2_partial/teacher_M2_partial.pkl"      # for the *_kd presets: <OUTPUT_DIR>/<EXP>/teacher_<EXP>.pkl of an earlier run

PRESETS = {
    "M1_adapter": dict(method="adapter", lr_backbone=1e-3, lr_head=2e-3, adapter_dim=64,
                       desc="Bottleneck adapters after every layer (backbone frozen)"),
    "M2_partial": dict(method="partial", lr_backbone=1e-4, lr_head=2e-3, train_top=5,
                       desc="Partial fine-tuning: only the top 5 of 20 layers (+ final norm) and the head are trained"),
    "M3_kd":      dict(method="full",    lr_backbone=1e-4, lr_head=2e-3, student_keep="odd", kd_alpha=0.5, kd_temp=2.0,
                       desc="Knowledge distillation: half-depth student (odd layers) learns from teacher statute logits"),
    "M4_lorma":   dict(method="lorma",   lr_backbone=1e-3, lr_head=2e-3, r=16, alpha=32, dropout=0.05, side="auto",
                       desc="(optional extra) LORMA multiplicative low-rank update on every linear layer"),
}

# ---- document representation (shared by all presets) ----
CHUNK_TOK, MAX_CHUNKS = 1024, 4        # tokens per chunk (<= model context) and chunks kept per case
CHUNK_POLICY = "uniform"               # "uniform" | "head_tail" | "last" | "first"
AGG = "label_attn"                     # "label_attn" | "attn" | "max" | "mean"
HEAD_DIM, HEAD_DROPOUT = 256, 0.2
LABEL_REFINE, REFINE_RANK = False, 32  # low-rank statute-interaction layer (ablation)
# ---- multi-label learning (shared) ----
LOSS = "bce_pw"                        # "bce" | "bce_pw" | "focal" | "asl"
POS_WEIGHT_CAP, FOCAL_GAMMA = 20.0, 2.0
SAMPLING, RARE_POWER = "rare_boost", 0.5      # "rare_boost" | "uniform"
THRESHOLD_MODE = "auto"                # "auto" | "global" | "per_label"
MIN_DEV_POS, SHRINK, MIN_PRED = 5, 0.5, 1
# ---- training budget (shared) ----
DOCS_PER_MICRO, GRAD_ACCUM = 1, 16
EPOCHS = 4
TRAIN_DOCS_PER_EPOCH = None            # None = every training case each epoch (the benchmark cell may set this)
AUTO_FIT_BUDGET, TARGET_TRAIN_MINUTES = True, 90
MAX_DEV_DOCS = None
WARMUP_FRAC, WEIGHT_DECAY, CLIP = 0.06, 0.01, 1.0
DEDUP_EXACT = True                     # drop exact duplicates inside train (same text and same statute set)
DROP_TRAIN_DEV_OVERLAP = True          # drop train cases whose text also appears in dev (keeps dev honest)
FINAL_RETRAIN = False                  # True = retrain on train+dev for the best epoch count (doubles the cost)
GRAD_CKPT = False
INFER_DOCS = 2
PROBE_MAX_DOCS = 300
TEACHER_SAVE_TRAIN_DOCS = 4000
SAVE_TEACHER = True
ALLOW_LARGER_STUDENT = False

if DEBUG:
    CHUNK_TOK, MAX_CHUNKS, HEAD_DIM = 96, 3, 32
    GRAD_ACCUM, EPOCHS, INFER_DOCS, PROBE_MAX_DOCS, TEACHER_SAVE_TRAIN_DOCS = 2, 3, 4, 20, 40
    TARGET_TRAIN_MINUTES = 5

EXP = PRESETS[EXPERIMENT]
EXP_DIR = f"{OUTPUT_DIR}/{EXPERIMENT}"
os.makedirs(EXP_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "", "| experiment:", EXPERIMENT, "-", EXP["desc"])

device: cuda | Tesla T4 | experiment: M2_partial - Partial fine-tuning: only the top 5 of 20 layers (+ final norm) and the head are trained


## 2. Phase-1 code, reused unchanged
`normalize_text`, the from-scratch `ByteBPE` tokenizer and the model architecture, copied from the Phase-1 notebooks so the checkpoint loads exactly.

In [2]:
CHAR_MAP = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-",
    "\u2212": "-", "\u2026": "...", "\u00a0": " ", "\u200b": "", "\ufeff": "", "\u00ad": "",
})
CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

def normalize_text(t: str, keep_newlines: bool) -> str:
    """Character and whitespace normalisation. Applied to train text AND test prompts."""
    t = CTRL.sub(" ", t.translate(CHAR_MAP))
    if keep_newlines:
        t = re.sub(r"[ \t]+", " ", t)
        t = re.sub(r" ?\n ?", "\n", t)
        t = re.sub(r"\n{3,}", "\n\n", t)
    else:
        t = re.sub(r"\s+", " ", t)
    return t.strip()

PRETOKEN_RE = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+""")

class ByteBPE:
    def __init__(self, merges=None):
        self.merges = [tuple(m) for m in (merges or [])]
        self.rank = {p: i for i, p in enumerate(self.merges)}
        self.vocab = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            self.vocab.append(self.vocab[a] + self.vocab[b])
        self.eos_id = len(self.vocab)
        self.vocab.append(EOS_PLACEHOLDER.encode())
        self.vocab_size = len(self.vocab)
        self._cache = {}

    # ---------- training ----------
    @staticmethod
    def train(chunk_counts, vocab_size, min_freq=3, log_every=2000):
        n_merges = vocab_size - 256 - 1                      # last id is EOS
        items = [(c.encode("utf-8"), f) for c, f in chunk_counts.items() if f >= min_freq]
        words = [list(b) for b, _ in items]
        freqs = [f for _, f in items]
        pair_cnt, where = defaultdict(int), defaultdict(set)
        for i, w in enumerate(words):
            for p in zip(w, w[1:]):
                pair_cnt[p] += freqs[i]; where[p].add(i)
        heap = [(-c, p) for p, c in pair_cnt.items()]
        heapq.heapify(heap)
        merges, t0 = [], time.time()
        while len(merges) < n_merges:
            best = None
            while heap:                                       # lazy deletion
                negc, p = heapq.heappop(heap)
                c = pair_cnt.get(p, 0)
                if c <= 0: continue
                if c != -negc:
                    heapq.heappush(heap, (-c, p)); continue
                best = p; break
            if best is None: break
            a, b = best; new_id = 256 + len(merges); merges.append(best)
            changed = set()
            for i in list(where[best]):
                w, f = words[i], freqs[i]
                out, j, n, hit = [], 0, len(w), False
                while j < n:
                    if j < n - 1 and w[j] == a and w[j + 1] == b:
                        out.append(new_id); j += 2; hit = True
                    else:
                        out.append(w[j]); j += 1
                if not hit: continue
                for q in zip(w, w[1:]): pair_cnt[q] -= f
                for q in zip(out, out[1:]):
                    pair_cnt[q] += f; where[q].add(i); changed.add(q)
                words[i] = out
            where.pop(best, None); pair_cnt.pop(best, None)
            for q in changed:
                if pair_cnt[q] > 0: heapq.heappush(heap, (-pair_cnt[q], q))
            if len(merges) % log_every == 0:
                print(f"  merges {len(merges):6d}/{n_merges} | {time.time() - t0:5.0f}s")
        return ByteBPE(merges)

    # ---------- encoding / decoding ----------
    def _encode_chunk(self, chunk):
        hit = self._cache.get(chunk)
        if hit is not None: return hit
        ids = list(chunk.encode("utf-8"))
        while len(ids) > 1:
            best_r, best_p = None, None
            for p in zip(ids, ids[1:]):
                r = self.rank.get(p)
                if r is not None and (best_r is None or r < best_r):
                    best_r, best_p = r, p
            if best_p is None: break
            a, b = best_p; new_id = 256 + best_r; out, j = [], 0
            while j < len(ids):
                if j < len(ids) - 1 and ids[j] == a and ids[j + 1] == b:
                    out.append(new_id); j += 2
                else:
                    out.append(ids[j]); j += 1
            ids = out
        if len(self._cache) > 3_000_000: self._cache.clear()
        self._cache[chunk] = ids
        return ids

    def encode(self, text):
        ids = []
        for chunk in PRETOKEN_RE.findall(text):
            ids.extend(self._encode_chunk(chunk))
        return ids

    def decode(self, ids):
        data = b"".join(self.vocab[i] for i in ids if i != self.eos_id)
        return data.decode("utf-8", errors="replace")

    def save(self, path):
        json.dump({"merges": self.merges, "eos": EOS_PLACEHOLDER}, open(path, "w"))

    @staticmethod
    def load(path):
        return ByteBPE(json.load(open(path))["merges"])

In [3]:
@dataclass
class HybridConfig:
    vocab_size: int
    n_ctx: int = 1280
    d_model: int = 768
    pattern: str = "MMMA" * 6        # M = Mamba-2 (SSD) layer, A = attention + SwiGLU layer
    # --- Mamba-2 / SSD layer ---
    expand: int = 2                  # d_inner = expand * d_model
    headdim: int = 64                # SSM head size P  (n_ssm_heads = d_inner / headdim)
    d_state: int = 64                # state size N per head
    d_conv: int = 4                  # causal depthwise conv width
    chunk: int = 64                  # SSD chunk length Q
    # --- attention layer ---
    n_head: int = 12
    n_kv_head: int = 4
    n_ffn: int = 2048
    rope_base: float = 10000.0

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.weight

class GatedRMSNorm(nn.Module):
    """Mamba-2 output norm: RMSNorm(y * silu(z))."""
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, y, z):
        y = y * F.silu(z)
        yf = y.float()
        return (yf * torch.rsqrt(yf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(y) * self.weight

# ------------------------------------------------------------------------------------------
# State-space duality (SSD) scan, pure PyTorch. Computes, for every head h,
#     s_t = exp(dt_t * A_h) * s_{t-1} + dt_t * (x_t outer B_t)      y_t = C_t . s_t
# in chunks of Q tokens: inside a chunk it is a masked "attention-like" matmul; across chunks only a
# small recurrence on the (P x N) states is needed. Decay math is fp32, big matmuls follow autocast.
# ------------------------------------------------------------------------------------------
def ssd_chunked(X, dt, A, Bm, Cm, Q):
    """X (B,T,H,P) | dt (B,T,H) fp32 | A (H,) fp32 <0 | Bm,Cm (B,T,N). Returns Y (B,T,H,P), final state (B,H,P,N) fp32."""
    Bsz, T, H, P = X.shape
    N = Bm.size(-1)
    pad = (-T) % Q
    if pad:                                                   # dt = 0 on padding => state is untouched
        X = F.pad(X, (0, 0, 0, 0, 0, pad)); dt = F.pad(dt, (0, 0, 0, pad))
        Bm = F.pad(Bm, (0, 0, 0, pad)); Cm = F.pad(Cm, (0, 0, 0, pad))
    Tp = T + pad; nC = Tp // Q
    Xc = (X * dt.to(X.dtype)[..., None]).view(Bsz, nC, Q, H, P)             # dt-scaled input
    Bc, Cc = Bm.reshape(Bsz, nC, Q, N), Cm.reshape(Bsz, nC, Q, N)
    a_cum = (dt * A).view(Bsz, nC, Q, H).permute(0, 3, 1, 2).cumsum(-1)     # (B,H,nC,Q) fp32, non-increasing

    # 1) inside each chunk: Y = (C B^T o L) X, with L[i,j] = exp(a_i - a_j) for i >= j
    tril = torch.tril(torch.ones(Q, Q, dtype=torch.bool, device=X.device))
    L = torch.exp((a_cum[..., :, None] - a_cum[..., None, :]).masked_fill(~tril, float("-inf")))
    G = torch.einsum("bcqn,bcsn->bcqs", Cc, Bc)                              # (B,nC,Q,Q)
    M = (G[:, None].float() * L).to(Xc.dtype)                                # (B,H,nC,Q,Q)
    Y_diag = torch.einsum("bhcqs,bcshp->bcqhp", M, Xc)

    # 2) the state each chunk would produce on its own
    decay_states = torch.exp(a_cum[..., -1:] - a_cum)                        # (B,H,nC,Q)
    Xw = Xc * decay_states.permute(0, 2, 3, 1)[..., None].to(Xc.dtype)
    states = torch.einsum("bcshp,bcsn->bchpn", Xw, Bc)                       # (B,nC,H,P,N)

    # 3) pass states across chunks (tiny recurrence, fp32)
    with torch.autocast(device_type=X.device.type, enabled=False):
        states = torch.cat([torch.zeros_like(states[:, :1]), states.float()], 1)     # (B,nC+1,H,P,N)
        cs = F.pad(a_cum[..., -1], (1, 0)).cumsum(-1)                                # (B,H,nC+1)
        seg = cs[..., :, None] - cs[..., None, :]
        keep = torch.tril(torch.ones(nC + 1, nC + 1, dtype=torch.bool, device=X.device))
        decay_chunk = torch.exp(seg.masked_fill(~keep, float("-inf")))
        new_states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states)
    prev_states, final_state = new_states[:, :-1], new_states[:, -1]

    # 4) contribution of everything before the chunk to each token inside it
    out_decay = torch.exp(a_cum).permute(0, 2, 3, 1)[..., None].to(Xc.dtype)         # (B,nC,Q,H,1)
    Y_off = torch.einsum("bcqn,bchpn->bcqhp", Cc, prev_states.to(Cc.dtype)) * out_decay
    return (Y_diag + Y_off).reshape(Bsz, Tp, H, P)[:, :T], final_state

class Mamba2Block(nn.Module):
    def __init__(self, c: HybridConfig):
        super().__init__()
        self.d_inner, self.P, self.N, self.K, self.Q = c.expand * c.d_model, c.headdim, c.d_state, c.d_conv, c.chunk
        assert self.d_inner % self.P == 0
        self.H = self.d_inner // self.P
        self.conv_dim = self.d_inner + 2 * self.N
        self.in_proj = nn.Linear(c.d_model, 2 * self.d_inner + 2 * self.N + self.H, bias=False)   # z | x B C | dt
        self.conv1d = nn.Conv1d(self.conv_dim, self.conv_dim, self.K, groups=self.conv_dim, padding=self.K - 1)
        dt = torch.exp(torch.rand(self.H) * (math.log(0.1) - math.log(1e-3)) + math.log(1e-3)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))            # softplus^-1(dt) with dt in [1e-3, 1e-1]
        self.A_log = nn.Parameter(torch.log(torch.empty(self.H).uniform_(1, 16)))
        self.D = nn.Parameter(torch.ones(self.H))
        self.norm = GatedRMSNorm(self.d_inner)
        self.out_proj = nn.Linear(self.d_inner, c.d_model, bias=False)

    def forward(self, x, valid=None, want_state=False):
        """Parallel (chunked) form for training and prompt prefill. valid: (B,T) bool, False = left-padding."""
        B, T, _ = x.shape
        z, xBC, dt = self.in_proj(x).split([self.d_inner, self.conv_dim, self.H], dim=-1)
        if valid is not None:
            xBC = xBC * valid[..., None].to(xBC.dtype)              # padding looks like the zero boundary of the conv
        xBC_t = xBC.transpose(1, 2)
        xBC = F.silu(self.conv1d(xBC_t)[..., :T]).transpose(1, 2)
        xs, Bm, Cm = xBC.split([self.d_inner, self.N, self.N], dim=-1)
        dtf = F.softplus(dt.float() + self.dt_bias)
        if valid is not None:
            dtf = dtf * valid[..., None].float()                    # dt = 0  =>  state passes through padding unchanged
        A = -torch.exp(self.A_log.float())
        X = xs.reshape(B, T, self.H, self.P)
        Y, state = ssd_chunked(X, dtf, A, Bm, Cm, self.Q)
        Y = Y + X * self.D.to(X.dtype)[None, None, :, None]
        out = self.out_proj(self.norm(Y.reshape(B, T, self.d_inner), z))
        if not want_state:
            return out, None
        conv_state = xBC_t[..., -(self.K - 1):]
        if conv_state.size(-1) < self.K - 1:
            conv_state = F.pad(conv_state, (self.K - 1 - conv_state.size(-1), 0))
        return out, (conv_state, state)

    def step(self, x, st):
        """Recurrent form for one token (B,1,D): constant memory, constant time."""
        conv_state, ssm = st
        z, xBC, dt = self.in_proj(x[:, 0]).split([self.d_inner, self.conv_dim, self.H], dim=-1)
        window = torch.cat([conv_state, xBC[..., None]], dim=-1)                    # (B, conv_dim, K)
        conv = (window.float() * self.conv1d.weight[:, 0, :].float()).sum(-1) + self.conv1d.bias.float()
        xs, Bm, Cm = F.silu(conv).split([self.d_inner, self.N, self.N], dim=-1)
        X = xs.view(-1, self.H, self.P)
        dtf = F.softplus(dt.float() + self.dt_bias)                                 # (B,H)
        dA = torch.exp(dtf * -torch.exp(self.A_log.float()))
        ssm = ssm * dA[..., None, None] + dtf[..., None, None] * X[..., None] * Bm[:, None, None, :]
        y = (ssm * Cm[:, None, None, :]).sum(-1) + self.D[None, :, None] * X        # (B,H,P)
        out = self.out_proj(self.norm(y.reshape(-1, 1, self.d_inner).to(z.dtype), z[:, None]))
        return out, (window[..., 1:], ssm)

def build_rope(head_dim, n_ctx, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(n_ctx).float(), inv)
    return freqs.cos(), freqs.sin()

def apply_rope(x, cos, sin):                                 # x: (B, H, T, hd); cos/sin: (*, 1, T, hd/2)
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)

class Attention(nn.Module):
    """Grouped-query attention with RoPE (only 1 in 4 layers is attention in the default pattern)."""
    def __init__(self, c: HybridConfig):
        super().__init__()
        self.nh, self.nkv, self.hd = c.n_head, c.n_kv_head, c.d_model // c.n_head
        self.q = nn.Linear(c.d_model, self.nh * self.hd, bias=False)
        self.k = nn.Linear(c.d_model, self.nkv * self.hd, bias=False)
        self.v = nn.Linear(c.d_model, self.nkv * self.hd, bias=False)
        self.o = nn.Linear(self.nh * self.hd, c.d_model, bias=False)
    def forward(self, x, cos, sin, mask=None, cache=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        new_cache = None
        if cache is not None:
            if cache[0] is not None:
                k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
            new_cache = (k, v)
        rep = self.nh // self.nkv
        if rep > 1:
            k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=(mask is None and cache is None))
        return self.o(y.transpose(1, 2).reshape(B, T, -1)), new_cache

class SwiGLU(nn.Module):
    def __init__(self, c: HybridConfig):
        super().__init__()
        self.gate = nn.Linear(c.d_model, c.n_ffn, bias=False)
        self.up = nn.Linear(c.d_model, c.n_ffn, bias=False)
        self.down = nn.Linear(c.n_ffn, c.d_model, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class MambaLayer(nn.Module):
    kind = "M"
    def __init__(self, c):
        super().__init__()
        self.norm, self.mixer = RMSNorm(c.d_model), Mamba2Block(c)
    def forward(self, x, cos, sin, mask, valid, want_cache):
        h, st = self.mixer(self.norm(x), valid, want_cache)
        return x + h, st
    def step(self, x, cos, sin, key_mask, cache):
        h, st = self.mixer.step(self.norm(x), cache)
        return x + h, st

class AttnLayer(nn.Module):
    kind = "A"
    def __init__(self, c):
        super().__init__()
        self.n1, self.attn = RMSNorm(c.d_model), Attention(c)
        self.n2, self.mlp = RMSNorm(c.d_model), SwiGLU(c)
    def forward(self, x, cos, sin, mask, valid, want_cache):
        h, cache = self.attn(self.n1(x), cos, sin, mask, (None, None) if want_cache else None)
        x = x + h
        return x + self.mlp(self.n2(x)), cache
    def step(self, x, cos, sin, key_mask, cache):
        h, cache = self.attn(self.n1(x), cos, sin, key_mask, cache)
        x = x + h
        return x + self.mlp(self.n2(x)), cache

class LegalHybrid(nn.Module):
    def __init__(self, c: HybridConfig):
        super().__init__()
        assert set(c.pattern) <= {"M", "A"}
        self.c = c
        self.tok_emb = nn.Embedding(c.vocab_size, c.d_model)
        self.blocks = nn.ModuleList([MambaLayer(c) if ch == "M" else AttnLayer(c) for ch in c.pattern])
        self.norm = RMSNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight                                   # weight tying
        cos, sin = build_rope(c.d_model // c.n_head, c.n_ctx, c.rope_base)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.grad_ckpt = False
        self.apply(self._init)
        for n, p in self.named_parameters():                                     # scaled residual init
            if n.endswith("out_proj.weight") or n.endswith("attn.o.weight") or n.endswith("mlp.down.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(len(c.pattern)))
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, valid=None, want_cache=False, last_only=False):
        """Training: model(x, y). Prompt prefill: model(idx, valid=left_pad_mask, want_cache=True, last_only=True)."""
        B, T = idx.shape
        x = self.tok_emb(idx)
        mask = None
        if valid is None:
            cos, sin = self.cos[:T][None, None], self.sin[:T][None, None]
        else:                                                                    # left-padded batch
            pos = (valid.long().cumsum(1) - 1).clamp(min=0)
            cos, sin = self.cos[pos][:, None], self.sin[pos][:, None]
            causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=idx.device))
            eye = torch.eye(T, dtype=torch.bool, device=idx.device)              # keeps pad rows finite
            mask = (causal[None] & (valid[:, None, :] | eye[None]))[:, None]
        caches = [] if want_cache else None
        for blk in self.blocks:
            if self.grad_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(
                    lambda x_, c_, s_, b=blk: b(x_, c_, s_, None, None, False)[0], x, cos, sin, use_reentrant=False)
            else:
                x, c_new = blk(x, cos, sin, mask, valid, want_cache)
                if want_cache: caches.append(c_new)
        x = self.norm(x)
        if last_only: x = x[:, -1:]
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss, caches

    def step(self, tok, pos, key_mask, caches):
        """One decoding step. tok (B,1); pos (B,) = position of tok; key_mask (B,1,1,L_cached+1) bool."""
        x = self.tok_emb(tok)
        cos, sin = self.cos[pos][:, None, None, :], self.sin[pos][:, None, None, :]
        new = []
        for blk, cache in zip(self.blocks, caches):
            x, c_new = blk.step(x, cos, sin, key_mask, cache)
            new.append(c_new)
        return self.head(self.norm(x)), new

## 3. Load the Phase-1 checkpoint and tokenizer

In [4]:
for p in (CKPT_PATH, TOK_PATH):
    if not os.path.exists(p):
        print("MISSING:", p, "\nFiles under /kaggle/input:")
        for root, _, files in os.walk("/kaggle/input"):
            for f in files: print("  ", os.path.join(root, f))
        raise FileNotFoundError(p)
ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
p1_cfg = HybridConfig(**ck["cfg"])
tokenizer = ByteBPE.load(TOK_PATH)
EOS_ID = tokenizer.eos_id
assert p1_cfg.vocab_size == tokenizer.vocab_size, \
    f"tokenizer/checkpoint mismatch: model vocab {p1_cfg.vocab_size} vs tokenizer {tokenizer.vocab_size}. Use the bpe_merges.json this checkpoint was trained with."
print(f"checkpoint step {ck.get('step')} | vocab {p1_cfg.vocab_size} | n_ctx {p1_cfg.n_ctx} | config: {ck['cfg']}")
print("tokenizer merges md5:", hashlib.md5(open(TOK_PATH, "rb").read()).hexdigest()[:12])
assert CHUNK_TOK <= p1_cfg.n_ctx, f"CHUNK_TOK {CHUNK_TOK} exceeds the model context {p1_cfg.n_ctx}"

def build_backbone():
    m = LegalHybrid(p1_cfg)
    m.load_state_dict(ck["model"])
    return m

def backbone_dim(m): return m.c.d_model
def embed(m): return m.tok_emb
def layers(m): return m.blocks
def final_norm(m): return m.norm

def run_blocks(m, x, mask, post=None):
    T = x.size(1)
    cos, sin = m.cos[:T][None, None], m.sin[:T][None, None]
    for i, blk in enumerate(m.blocks):
        if GRAD_CKPT and m.training:
            x = torch.utils.checkpoint.checkpoint(lambda x_, b=blk: b(x_, cos, sin, None, None, False)[0], x, use_reentrant=False)
        else:
            x, _ = blk(x, cos, sin, None, None, False)
        if post is not None: x = x + post[i](x)
    return m.norm(x)

def prune_layers(m, keep):
    """Keep only the listed layers: a half-depth student that still starts from Phase-1 weights."""
    m.blocks = nn.ModuleList([m.blocks[i] for i in keep])
    return m

def inject_prefix(backbone, P):
    raise NotImplementedError("prefix tuning is implemented for the encoder-decoder notebook (its attention is full multi-head); use another preset here")

_probe = build_backbone().eval()
print(f"Phase-1 backbone used here: {sum(p.numel() for p in _probe.parameters()) / 1e6:.1f}M parameters, {len(layers(_probe))} layers (loaded OK)")
del _probe

checkpoint step 1000 | vocab 32000 | n_ctx 1280 | config: {'vocab_size': 32000, 'n_ctx': 1280, 'd_model': 768, 'pattern': 'MMMAMMMAMMMAMMMAMMMA', 'expand': 2, 'headdim': 64, 'd_state': 64, 'd_conv': 4, 'chunk': 64, 'n_head': 12, 'n_kv_head': 4, 'n_ffn': 2048, 'rope_base': 10000.0}
tokenizer merges md5: 16ee00bde3c5
Phase-1 backbone used here: 111.0M parameters, 20 layers (loaded OK)


## 4. Phase-5 data: load, parse robustly, audit
`train.json` may repeat ids (the single- and multi-instance data can be merged), so every case gets a unique internal key, duplicates are diagnosed, exact copies are removed and train cases that also occur in dev are dropped (they would inflate the dev score). The label space is the set of statute IDs in train+dev. Output ids keep their original type.

In [5]:
from huggingface_hub import HfApi, hf_hub_download, login
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("capstone-phase1-token")
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
_repo_files = HfApi().list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)

def fetch(*must_contain):
    hits = [f for f in _repo_files if all(s.lower() in f.lower() for s in must_contain)]
    if not hits:
        raise FileNotFoundError(f"{must_contain} not found. Repo has: {_repo_files[:60]}")
    return hf_hub_download(HF_DATASET_REPO, hits[0], repo_type="dataset", token=HF_TOKEN)
print([f for f in _repo_files if "phase_5" in f.lower()])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


['Phase_5_LSI/Test/test.json', 'Phase_5_LSI/Train/dev.json', 'Phase_5_LSI/Train/train.json']


In [6]:
P5 = {"train": fetch("phase_5", "train.json"), "dev": fetch("phase_5", "dev.json"), "test": fetch("phase_5", "test.json")}

def load_json(p):
    with open(p, encoding="utf-8") as f: return json.load(f)

def to_ids(raw, cid):
    vals = raw if isinstance(raw, (list, tuple, set)) else [raw]
    try: return sorted({int(v) for v in vals})
    except (TypeError, ValueError): raise ValueError(f"case {cid}: statute labels must be integers, got {raw!r}")

def parse_cases(obj, labelled):
    if isinstance(obj, dict) and len(obj) == 1 and isinstance(next(iter(obj.values())), list): obj = next(iter(obj.values()))
    if isinstance(obj, dict): items = [(k, dict(v) if isinstance(v, dict) else {"text": v}) for k, v in obj.items()]
    else: items = [(r.get("id", r.get("case_id", r.get("doc_id", i))), r) for i, r in enumerate(obj)]
    docs, seen = [], Counter()
    for cid, r in items:
        y = None
        if labelled:
            raw = next((r[k] for k in ("labels", "label", "statutes", "statute_ids", "y") if k in r), None)
            assert raw is not None, f"case {cid}: no label field among {list(r)}"
            y = to_ids(raw, cid)
        base = str(cid); seen[base] += 1
        key = base if seen[base] == 1 else f"{base}#{seen[base]}"      # internal keys stay unique even if ids repeat
        docs.append(dict(key=key, id=cid, text=str(r.get("text", r.get("case_text", ""))), y=y, yi=None, ids=None))
    return docs

RAW = {"train": parse_cases(load_json(P5["train"]), True), "dev": parse_cases(load_json(P5["dev"]), True), "test": parse_cases(load_json(P5["test"]), False)}
for sp in RAW:
    for d in RAW[sp]: d["h"] = hashlib.md5(d["text"].encode("utf-8", "ignore")).hexdigest()

# ---- duplicate diagnostics ----
for sp, ds in RAW.items():
    groups = defaultdict(list)
    for d in ds: groups[str(d["id"])].append(d)
    dup = {k: v for k, v in groups.items() if len(v) > 1}
    if dup:
        same = sum(1 for v in dup.values() if len({(x["h"], tuple(x["y"] or [])) for x in v}) == 1)
        distinct = sum(1 for v in dup.values() if len({x["h"] for x in v}) > 1)
        print(f"{sp}: {len(dup)} ids occur more than once ({sum(len(v) for v in dup.values())} cases) -> exact copies: {same} | DIFFERENT cases sharing an id: {distinct}")
assert len({str(d["id"]) for d in RAW["test"]}) == len(RAW["test"]), \
    "duplicate ids in test.json: the submission needs exactly one prediction per test id, so check with the organisers how repeated ids must be handled"
dev_hashes = {d["h"] for d in RAW["dev"]}; seen_h = set(); clean = []; removed = Counter(); n0 = len(RAW["train"])
for d in RAW["train"]:
    if DROP_TRAIN_DEV_OVERLAP and d["h"] in dev_hashes: removed["same text as a dev case"] += 1; continue
    if DEDUP_EXACT and (d["h"], tuple(d["y"])) in seen_h: removed["exact duplicate inside train"] += 1; continue
    seen_h.add((d["h"], tuple(d["y"]))); clean.append(d)
RAW["train"] = clean
print(f"train cleaning: {n0} -> {len(clean)} cases (removed: {dict(removed) if removed else 'nothing'})")
test_h = {d["h"] for d in RAW["test"]}
print(f"overlap check: train cases whose text also appears in test: {sum(d['h'] in test_h for d in RAW['train'])} | dev cases in test: {sum(d['h'] in test_h for d in RAW['dev'])}")

# ---- label space ----
LABEL_IDS = sorted({l for sp in ("train", "dev") for d in RAW[sp] for l in d["y"]})
L2I = {l: i for i, l in enumerate(LABEL_IDS)}; L = len(LABEL_IDS)
for sp in ("train", "dev"):
    for d in RAW[sp]: d["yi"] = [L2I[l] for l in d["y"]]
train_freq = np.zeros(L, dtype=np.int64)
for d in RAW["train"]:
    for j in d["yi"]: train_freq[j] += 1
dev_only = [LABEL_IDS[j] for j in range(L) if train_freq[j] == 0]
print(f"\nlabel space: {L} statute IDs (min {LABEL_IDS[0]}, max {LABEL_IDS[-1]}) | in train: {int((train_freq > 0).sum())} | only in dev (unlearnable in phase A): {len(dev_only)}")
for sp in ("train", "dev"):
    nl = [len(d["y"]) for d in RAW[sp]]; print(f"{sp:5s}: {len(nl)} cases, statutes per case: mean {np.mean(nl):.2f}, median {int(np.median(nl))}, max {max(nl)}")
fq = train_freq[train_freq > 0]
print(f"train frequency per statute: median {int(np.median(fq))}, max {int(fq.max())} | statutes with <=5 train cases: {int((fq <= 5).sum())} ({(fq <= 5).mean():.0%} of seen statutes), <=20: {int((fq <= 20).sum())}")
print("split | cases | median chars | max chars")
for sp, ds in RAW.items():
    ln = [len(d["text"]) for d in ds]; print(f"{sp:5s} | {len(ds):6d} | {int(np.median(ln)):12d} | {max(ln)}")
print("test id type:", type(RAW["test"][0]["id"]).__name__, "| sample id:", RAW["test"][0]["id"])
rs = random.Random(0); sample = rs.sample(RAW["train"], min(150, len(RAW["train"])))
tk = [len(tokenizer.encode(normalize_text(d["text"], False))) for d in sample]
print(f"tokens per case (sample of {len(sample)}): median {int(np.median(tk))}, p90 {int(np.percentile(tk, 90))}, max {max(tk)} | chunks of {CHUNK_TOK}: median {math.ceil(np.median(tk) / CHUNK_TOK)}, "
      f"cases that exceed {MAX_CHUNKS} chunks: {np.mean([t > MAX_CHUNKS * CHUNK_TOK for t in tk]):.0%}")

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

train cleaning: 42750 -> 42312 cases (removed: {'same text as a dev case': 210, 'exact duplicate inside train': 228})
overlap check: train cases whose text also appears in test: 106 | dev cases in test: 96

label space: 100 statute IDs (min 0, max 99) | in train: 100 | only in dev (unlearnable in phase A): 0
train: 42312 cases, statutes per case: mean 3.70, median 3, max 31
dev  : 10181 cases, statutes per case: mean 3.92, median 3, max 28
train frequency per statute: median 669, max 11565 | statutes with <=5 train cases: 0 (0% of seen statutes), <=20: 0
split | cases | median chars | max chars
train |  42312 |         2732 | 535964
dev   |  10181 |         3418 | 442434
test  |  13019 |         3368 | 820502
test id type: str | sample id: 100002997
tokens per case (sample of 150): median 765, p90 3607, max 12511 | chunks of 1024: median 1, cases that exceed 4 chunks: 7%


## 5. Chunking: which parts of a case the model sees
Chunk boundaries are aligned from the **end** of the document. Policies: `uniform` (evenly spaced across the case: default, since statute citations can be anywhere), `head_tail`, `last`, `first`. A case with at most `MAX_CHUNKS` chunks is always used whole.

In [7]:
def get_ids(d):
    if d["ids"] is None:                                   # lazy: only cases that are actually used get tokenised
        ids = tokenizer.encode(normalize_text(d["text"], False))
        d["ids"] = np.array(ids if ids else [EOS_ID], dtype=np.int32)
    return d["ids"]

def select_chunks(n, policy, M):
    k = max(1, math.ceil(n / CHUNK_TOK))
    bounds = [(max(0, n - (k - j) * CHUNK_TOK), n - (k - j - 1) * CHUNK_TOK) for j in range(k)]
    if k <= M: idx = list(range(k))
    elif policy == "first": idx = list(range(M))
    elif policy == "last": idx = list(range(k - M, k))
    elif policy == "head_tail": h = (M + 1) // 2; idx = list(range(h)) + list(range(k - (M - h), k))
    elif policy == "uniform": idx = sorted(set(np.linspace(0, k - 1, M).round().astype(int).tolist()))
    else: raise ValueError(policy)
    return [(*bounds[j], j / max(k - 1, 1)) for j in idx], k

def collate(docs, policy=None, M=None):
    policy, M = policy or CHUNK_POLICY, M or MAX_CHUNKS
    seqs, owner, rank, bucket = [], [], [], []
    for di, d in enumerate(docs):
        ids = get_ids(d); sel, d["_k"] = select_chunks(len(ids), policy, M)
        for r, (st, en, rel) in enumerate(sel):
            seqs.append(ids[st:en]); owner.append(di); rank.append(r); bucket.append(min(int(rel * 8), 7))
    N, T = len(seqs), max(len(s) for s in seqs)
    ids_t = torch.full((N, T), EOS_ID, dtype=torch.long); lens = torch.tensor([len(s) for s in seqs])
    for i, s in enumerate(seqs): ids_t[i, :len(s)] = torch.from_numpy(s.astype(np.int64))
    nch = torch.bincount(torch.tensor(owner), minlength=len(docs))
    Y = torch.zeros(len(docs), L); teach = torch.zeros(len(docs), L)
    for i, d in enumerate(docs):
        if d["yi"] is not None: Y[i, d["yi"]] = 1.0
        if d.get("teacher") is not None: teach[i] = torch.from_numpy(d["teacher"])
    return dict(ids=ids_t.to(device), tok_mask=(torch.arange(T)[None] < lens[:, None]).to(device), lens=lens.to(device),
                owner=torch.tensor(owner).to(device), rank=torch.tensor(rank).to(device), bucket=torch.tensor(bucket).to(device),
                nch=nch.to(device), nch_cpu=nch, Y=Y.to(device), teach=teach.to(device))

for pol in ("first", "last", "head_tail", "uniform"):
    sel, k = select_chunks(9 * CHUNK_TOK - 17, pol, MAX_CHUNKS)
    print(f"{pol:9s}: case of 9 chunks keeps chunk indices {[round(s[2] * (k - 1)) for s in sel]}")

first    : case of 9 chunks keeps chunk indices [0, 1, 2, 3]
last     : case of 9 chunks keeps chunk indices [5, 6, 7, 8]
head_tail: case of 9 chunks keeps chunk indices [0, 1, 7, 8]
uniform  : case of 9 chunks keeps chunk indices [0, 3, 5, 8]


## 6. The model: Phase-1 backbone + adaptation technique + label-wise attention head
* **full / partial:** every backbone weight / only the top `train_top` layers (+ final norm). **adapter:** a bottleneck adapter after every layer. **prompt:** learned soft tokens prepended to every chunk. **prefix:** learned key/value vectors per encoder layer.
* **lorma:** multiplicative low-rank update (see the note at the top). **qlora:** frozen 4-bit NF4 base weights + LoRA.
* **Knowledge distillation:** the student keeps the odd layers of the Phase-1 model and is fully fine-tuned with `(1-α)·loss(gold) + α·T²·BCE(σ(z_s/T), σ(z_t/T))` on the statute logits.
* **Head:** chunk vector -> linear + relative-position embedding -> label-wise attention over chunks (one query per statute) -> per-statute scorer; the output bias is initialised to the statute's log-odds prior.

In [8]:
class LoRMALinear(nn.Module):
    """Multiplicative low-rank adaptation (my reconstruction of LORMA; verify against reference [12]).
    side='out': y = (I + s B A)(W0 x)     side='in': y = W0((I + s B A) x)     B = 0 at init => identity update."""
    def __init__(self, base, r, alpha, dropout, side="auto"):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.side = side if side != "auto" else ("out" if base.out_features <= base.in_features else "in")
        d = base.out_features if self.side == "out" else base.in_features
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r)); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        if self.side == "out":
            y = self.base(x)
            return y + (self.drop(y) @ self.A.t() @ self.B.t()) * self.scale
        return self.base(x + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale)

NF4 = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
                    -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141506195, 0.24611230194568634, 0.33791524171829224,
                    0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])      # NF4 code values (verify against the QLoRA paper)
QBLOCK = 64

class QLoRALinear(nn.Module):
    """QLoRA from scratch: the frozen weight is stored as packed 4-bit NF4 codes + one fp16 absmax per block of 64 weights,
    de-quantised on the fly; LoRA matrices are trained on top."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        W = base.weight.data.float(); self.shape = tuple(W.shape); n = W.numel(); pad = (-n) % QBLOCK
        flat = F.pad(W.flatten(), (0, pad)).view(-1, QBLOCK)
        absmax = flat.abs().amax(1).clamp(min=1e-8)
        codes = torch.bucketize(flat / absmax[:, None], (NF4[1:] + NF4[:-1]) / 2).to(torch.uint8)       # nearest NF4 level
        self.register_buffer("packed", (codes[:, 0::2] | (codes[:, 1::2] << 4)).contiguous())
        self.register_buffer("absmax", absmax.half()); self.n = n
        self.register_buffer("nf4", NF4.clone(), persistent=False)
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, self.shape[1])); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.shape[0], r)); self.drop = nn.Dropout(dropout)
        self.in_features, self.out_features = self.shape[1], self.shape[0]
    def dequant(self):
        codes = torch.stack([self.packed & 0xF, self.packed >> 4], -1).reshape(self.packed.size(0), QBLOCK).long()
        return (self.nf4[codes] * self.absmax.float()[:, None]).flatten()[:self.n].view(self.shape)
    def forward(self, x):
        return F.linear(x, self.dequant()) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale
    def quant_bytes(self): return self.packed.numel() + 2 * self.absmax.numel()

class Adapter(nn.Module):
    def __init__(self, d, r):
        super().__init__()
        self.down, self.up = nn.Linear(d, r), nn.Linear(r, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x): return self.up(F.gelu(self.down(x)))

LINEAR_TARGETS = {"in_proj", "out_proj", "q", "k", "v", "o", "gate", "up", "down"}
def inject(module, make):
    n = 0
    for name, child in list(module.named_children()):
        new = make(name, child) if isinstance(child, nn.Linear) else None
        if new is not None: setattr(module, name, new); n += 1
        else: n += inject(child, make)
    return n

def layer_index(name):
    m = re.search(r"(?:^|\.)(?:blocks|enc_blocks)\.(\d+)\.", name)
    return int(m.group(1)) if m else -1

PRIOR = np.clip((train_freq + 1.0) / (len(RAW["train"]) + 2.0), 1e-4, 0.5)

class Phase5Model(nn.Module):
    def __init__(self, backbone, exp):
        super().__init__()
        self.backbone, self.method, self.exp = backbone, exp["method"], exp
        d = backbone_dim(backbone); self.adapters = None; self.n_wrapped = 0; self.P = 0; self.frozen_bytes = 0
        if self.method != "full":
            for p in backbone.parameters(): p.requires_grad = False
        L_ = len(layers(backbone))
        if self.method == "lorma":
            self.n_wrapped = inject(backbone, lambda n, c: LoRMALinear(c, exp["r"], exp["alpha"], exp["dropout"], exp.get("side", "auto")) if n in LINEAR_TARGETS else None)
        elif self.method == "qlora":
            self.n_wrapped = inject(backbone, lambda n, c: QLoRALinear(c, exp["r"], exp["alpha"], exp["dropout"]) if n in LINEAR_TARGETS else None)
            self.frozen_bytes = sum(m.quant_bytes() for m in backbone.modules() if isinstance(m, QLoRALinear))
        elif self.method == "adapter":
            self.adapters = nn.ModuleList([Adapter(d, exp["adapter_dim"]) for _ in range(L_)])
        elif self.method == "partial":
            for i in range(L_ - exp["train_top"], L_):
                for p in layers(backbone)[i].parameters(): p.requires_grad = True
            for p in final_norm(backbone).parameters(): p.requires_grad = True
        elif self.method == "prompt":
            self.P = exp["prompt_len"]; E = embed(backbone)
            self.prompt = nn.Parameter(E.weight[torch.randint(256, min(E.num_embeddings, 4000), (self.P,))].detach().clone())
        elif self.method == "prefix":
            self.n_wrapped = inject_prefix(backbone, exp["prefix_len"])
        H = HEAD_DIM
        self.proj = nn.Linear(2 * d, H); self.pos = nn.Embedding(8, H); self.drop = nn.Dropout(HEAD_DROPOUT)
        if AGG == "label_attn": self.q = nn.Parameter(torch.randn(L, H) * 0.02); self.w_out = nn.Parameter(torch.randn(L, H) * 0.02)
        else:
            if AGG == "attn": self.att_in, self.att_out = nn.Linear(H, H), nn.Linear(H, 1, bias=False)
            self.cls = nn.Linear(H, L)
        self.bias = nn.Parameter(torch.tensor(np.log(PRIOR / (1 - PRIOR)), dtype=torch.float32))
        if LABEL_REFINE: self.ru, self.rv = nn.Parameter(torch.randn(L, REFINE_RANK) * 0.02), nn.Parameter(torch.zeros(L, REFINE_RANK))

    def hidden(self, b):
        m = self.backbone; x = embed(m)(b["ids"]); mask = b["tok_mask"]
        if self.P:
            x = torch.cat([self.prompt.unsqueeze(0).expand(x.size(0), -1, -1).to(x.dtype), x], 1)
            mask = F.pad(mask, (self.P, 0), value=True)
        h = run_blocks(m, x, mask, self.adapters)
        return h[:, self.P:] if self.P else h

    def forward(self, b):
        h = self.hidden(b); N = h.size(0); v = b["tok_mask"].to(h.dtype)
        mean = (h * v[..., None]).sum(1) / v.sum(1, keepdim=True)
        last = h[torch.arange(N, device=h.device), b["lens"] - 1]
        cv = F.gelu(self.proj(torch.cat([mean, last], -1))) + self.pos(b["bucket"])
        Dn, C = b["nch"].numel(), int(b["nch_cpu"].max()); H = cv.size(-1)
        x = cv.new_zeros(Dn, C, H).index_put((b["owner"], b["rank"]), cv)
        cmask = torch.arange(C, device=h.device)[None] < b["nch"][:, None]
        w = None
        if AGG == "label_attn":
            s = torch.einsum("dch,lh->dcl", x, self.q).float() / math.sqrt(H)
            w = torch.softmax(s.masked_fill(~cmask[..., None], -1e4), 1).to(x.dtype)           # softmax over chunks, per statute
            ctx = torch.einsum("dcl,dch->dlh", w, x); logits = (self.drop(ctx) * self.w_out).sum(-1) + self.bias
        elif AGG == "attn":
            s = self.att_out(torch.tanh(self.att_in(x))).squeeze(-1).masked_fill(~cmask, -1e4)
            w = torch.softmax(s.float(), -1).to(x.dtype); logits = self.cls(self.drop((w[..., None] * x).sum(1))) + self.bias
        elif AGG == "max":
            lc = self.cls(self.drop(x)).masked_fill(~cmask[..., None], -1e4); logits = lc.max(1).values + self.bias
        else:
            logits = self.cls(self.drop((x * cmask[..., None]).sum(1) / cmask.sum(1, keepdim=True))) + self.bias
        if LABEL_REFINE: logits = logits + (torch.sigmoid(logits.float()).to(logits.dtype) @ self.ru) @ self.rv.t()
        return logits, w

def build_model():
    bb = build_backbone()
    if EXP.get("student_keep") == "odd": prune_layers(bb, list(range(1, len(layers(bb)), 2)))
    return Phase5Model(bb, EXP).to(device)

def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)

_m = build_model(); TOTAL_P, TRAIN_P = count_params(_m); FROZEN_MB = _m.frozen_bytes / 1e6
print(f"{EXPERIMENT}: total {TOTAL_P / 1e6:.2f}M params | trainable at start {TRAIN_P / 1e6:.3f}M ({100 * TRAIN_P / TOTAL_P:.2f}%) | {len(layers(_m.backbone))} backbone layers | {L} statutes"
      + (f" | {_m.n_wrapped} layers adapted" if _m.n_wrapped else ""))
if EXP["method"] == "qlora":
    ql = next(m for m in _m.backbone.modules() if isinstance(m, QLoRALinear)); ref = build_backbone()
    ref_w = [m.weight for m in ref.modules() if isinstance(m, nn.Linear) and m.weight.shape == ql.shape][0]
    err = ((ql.dequant() - ref_w).norm() / ref_w.norm()).item()
    all_lin = sum(m.weight.numel() for m in ref.modules() if isinstance(m, nn.Linear)) * 4 / 1e6
    print(f"QLoRA: relative quantisation error of a sample layer {err:.3f} | quantised linear weights {FROZEN_MB:.0f} MB (fp32 would be about {all_lin:.0f} MB)"); del ref
del _m

M2_partial: total 111.48M params | trainable at start 24.033M (21.56%) | 20 backbone layers | 100 statutes


## 7. Losses, metrics, thresholds
* **Metrics:** macro-F1 over statutes present in the dev gold (primary), also over the union of gold and predicted statutes and over all statutes in the label space; micro-F1, subset accuracy, macro precision/recall.
* **Thresholds:** `global` (one value for all statutes), `per_label` (a value per statute tuned on dev, used only for statutes with at least `MIN_DEV_POS` dev positives and shrunk by `SHRINK` towards the global value). `auto` picks by a **2-fold estimate** (tune on one half of dev, score on the other half), because per-label thresholds tuned and scored on the same dev set are optimistic.
* **Fallback:** a case with no statute above threshold receives its single most likely statute.

In [9]:
THR_GRID = np.round(np.concatenate([np.linspace(0.02, 0.2, 10), np.linspace(0.25, 0.9, 14)]), 3)

def prf(Y, P):
    tp = (Y & P).sum(0).astype(np.float64); fp = (~Y & P).sum(0).astype(np.float64); fn = (Y & ~P).sum(0).astype(np.float64)
    return tp, fp, fn
def f1_vec(tp, fp, fn):
    den = 2 * tp + fp + fn; return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)

def ml_scores(Y, P):
    """Y, P: boolean (N, L)."""
    tp, fp, fn = prf(Y, P); f1 = f1_vec(tp, fp, fn); pr = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0); rc = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    present = Y.sum(0) > 0; union = present | (P.sum(0) > 0)
    mic = f1_vec(np.array([tp.sum()]), np.array([fp.sum()]), np.array([fn.sum()]))[0]
    return dict(macro_f1=float(f1[present].mean()) if present.any() else 0.0, macro_p=float(pr[present].mean()) if present.any() else 0.0,
                macro_r=float(rc[present].mean()) if present.any() else 0.0, macro_f1_union=float(f1[union].mean()) if union.any() else 0.0,
                macro_f1_all=float(f1.mean()), micro_f1=float(mic), subset_acc=float((Y == P).all(1).mean()), per_label_f1=f1,
                avg_pred=float(P.sum(1).mean()), avg_gold=float(Y.sum(1).mean()))

def apply_thr(probs, thr):
    P = probs >= np.asarray(thr)[None, :] if np.ndim(thr) else probs >= thr
    for i in np.where(P.sum(1) < MIN_PRED)[0]:                         # every case gets at least MIN_PRED statutes
        P[i, np.argsort(-probs[i])[:MIN_PRED]] = True
    return P

def best_global(Y, probs):
    best = (0.5, -1.0)
    for t in THR_GRID:
        f = ml_scores(Y, apply_thr(probs, float(t)))["macro_f1"]
        if f > best[1]: best = (float(t), f)
    return best

def per_label_thr(Y, probs, t_glob):
    G = len(THR_GRID); F_ = np.zeros((G, probs.shape[1]))
    for g, t in enumerate(THR_GRID):
        tp, fp, fn = prf(Y, probs >= t); F_[g] = f1_vec(tp, fp, fn)
    best_t = THR_GRID[F_.argmax(0)]; thr = np.full(probs.shape[1], t_glob)
    ok = Y.sum(0) >= MIN_DEV_POS
    thr[ok] = SHRINK * best_t[ok] + (1 - SHRINK) * t_glob
    return thr

def choose_thresholds(Y, probs, mode):
    t_glob, f_glob = best_global(Y, probs); thr_pl = per_label_thr(Y, probs, t_glob)
    info = dict(global_thr=t_glob, global_f1_in_sample=f_glob, per_label_f1_in_sample=ml_scores(Y, apply_thr(probs, thr_pl))["macro_f1"])
    if len(Y) >= 4:                                                    # honest 2-fold estimate
        idx = np.arange(len(Y)); res = {"global": [], "per_label": []}
        for a, b in ((idx % 2 == 0, idx % 2 == 1), (idx % 2 == 1, idx % 2 == 0)):
            tg, _ = best_global(Y[a], probs[a]); tp_ = per_label_thr(Y[a], probs[a], tg)
            res["global"].append(ml_scores(Y[b], apply_thr(probs[b], tg))["macro_f1"]); res["per_label"].append(ml_scores(Y[b], apply_thr(probs[b], tp_))["macro_f1"])
        info.update(honest_global=float(np.mean(res["global"])), honest_per_label=float(np.mean(res["per_label"])))
    chosen = mode if mode != "auto" else ("per_label" if info.get("honest_per_label", 0) > info.get("honest_global", 1) else "global")
    info["mode"] = chosen
    return (thr_pl if chosen == "per_label" else t_glob), info

def multilabel_loss(logits, Y, pw):
    z = logits.float()
    if LOSS == "bce": return F.binary_cross_entropy_with_logits(z, Y)
    if LOSS == "bce_pw": return F.binary_cross_entropy_with_logits(z, Y, pos_weight=pw)
    if LOSS == "focal":
        p = torch.sigmoid(z); ce = F.binary_cross_entropy_with_logits(z, Y, reduction="none"); pt = p * Y + (1 - p) * (1 - Y)
        return ((1 - pt) ** FOCAL_GAMMA * ce).mean()
    if LOSS == "asl":                                                   # asymmetric loss: negatives down-weighted and probability-shifted
        p = torch.sigmoid(z); pm = (p - 0.05).clamp(min=0)
        return -(Y * torch.log(p.clamp(min=1e-8)) + (1 - Y) * (pm ** 4) * torch.log((1 - pm).clamp(min=1e-8))).mean()
    raise ValueError(LOSS)

def kd_loss(logits, teach, T):
    return F.binary_cross_entropy_with_logits(logits.float() / T, torch.sigmoid(teach / T)) * T * T

## 8. Evaluation, teacher loading and training machinery
* Every epoch the dev set is scored with the **best global threshold** (macro-F1) and the best epoch is restored; final thresholds are chosen afterwards (section 9).
* Each epoch draws `TRAIN_DOCS_PER_EPOCH` cases (all if `None`); with `SAMPLING="rare_boost"` a case's chance of being drawn grows with the rarity of its rarest statute.
* `unfreeze`-style gradual schedules are not used in Phase 5; partial fine-tuning, adapters, prompts, prefixes, LORMA and QLoRA train a fixed set of parameters.

In [10]:
@torch.no_grad()
def predict(model, docs, policy=None, M=None):
    """-> probabilities (n, L) and per-case attention weights (list of (n_chunks, L) or (n_chunks,) arrays)."""
    model.eval(); out, atts, logit_list = [], [], []
    for i in range(0, len(docs), INFER_DOCS):
        chunk = docs[i:i + INFER_DOCS]; b = collate(chunk, policy, M)
        with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
            logits, w = model(b)
        logit_list.append(logits.float().cpu().numpy()); out.append(torch.sigmoid(logits.float()).cpu().numpy())
        if w is not None:
            w = w.float().cpu().numpy(); atts += [w[j, :int(b["nch_cpu"][j])] for j in range(len(chunk))]
    return np.concatenate(out), atts, np.concatenate(logit_list)

def gold_matrix(docs):
    Y = np.zeros((len(docs), L), dtype=bool)
    for i, d in enumerate(docs): Y[i, d["yi"]] = True
    return Y

def evaluate(model, docs):
    t0 = time.time(); probs, atts, logit = predict(model, docs); Y = gold_matrix(docs); t, f = best_global(Y, probs)
    return f, t, probs, atts, logit, time.time() - t0

TEACHER = None
if EXP.get("kd_alpha") is not None:
    with open(TEACHER_PKL, "rb") as f: TEACHER = pickle.load(f)
    assert TEACHER["label_ids"] == LABEL_IDS, "the teacher was produced with a different statute label space (different train/dev data or cleaning)"
    print(f"teacher: {TEACHER['meta']['experiment']} ({TEACHER['meta']['total_params'] / 1e6:.1f}M params, dev macro-F1 {TEACHER['meta']['dev_macro_f1']:.4f}), "
          f"{len(TEACHER['train'])} training cases with teacher outputs")

def class_pos_weight(docs):
    pos = np.zeros(L)
    for d in docs:
        for j in d["yi"]: pos[j] += 1
    neg = len(docs) - pos
    return torch.tensor(np.clip(np.sqrt(neg / np.maximum(pos, 1.0)), 1.0, POS_WEIGHT_CAP), dtype=torch.float32, device=device)

def make_optimizer(model):
    bb = [(n, p) for n, p in model.named_parameters() if p.requires_grad and n.startswith(("backbone.", "adapters.", "prompt"))]
    head = [(n, p) for n, p in model.named_parameters() if p.requires_grad and not n.startswith(("backbone.", "adapters.", "prompt"))]
    groups = []
    for lst, lr in ((bb, EXP["lr_backbone"]), (head, EXP["lr_head"])):
        dec = [p for n, p in lst if p.dim() >= 2]; nod = [p for n, p in lst if p.dim() < 2]
        for ps, wd in ((dec, WEIGHT_DECAY), (nod, 0.0)):
            if ps: groups.append({"params": ps, "lr": lr, "base_lr": lr, "weight_decay": wd})
    return torch.optim.AdamW(groups, betas=(0.9, 0.98))

def snapshot(model):
    if EXP["method"] in ("full", "partial"): return {n: v.detach().cpu().clone() for n, v in model.state_dict().items()}
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {n: v.detach().cpu().clone() for n, v in model.state_dict().items() if n in keep}

def sample_epoch(train_docs, n_ep, rng):
    if SAMPLING == "uniform" or n_ep >= len(train_docs) and SAMPLING == "uniform": return rng.sample(train_docs, n_ep)
    w = [max((max(float(train_freq[j]), 1.0) ** -RARE_POWER) for j in d["yi"]) for d in train_docs]
    keys = [rng.random() ** (1.0 / wi) for wi in w]                      # weighted sampling without replacement (Efraimidis-Spirakis)
    order = sorted(range(len(train_docs)), key=lambda i: -keys[i])[:n_ep]; rng.shuffle(order)
    return [train_docs[i] for i in order]

def run_training(train_docs, dev_docs, epochs, tag, use_kd, docs_per_epoch):
    model = build_model(); pw = class_pos_weight(train_docs); opt = make_optimizer(model)
    scaler = torch.amp.GradScaler(device, enabled=device == "cuda")
    n_ep = min(len(train_docs), docs_per_epoch or len(train_docs)); per_step = DOCS_PER_MICRO * GRAD_ACCUM
    steps_per_epoch = max(1, n_ep // per_step); total = steps_per_epoch * epochs; warm = max(1, int(WARMUP_FRAC * total)); step = 0
    rng = random.Random(SEED); best = dict(f1=-1.0, epoch=0, state=None); hist = []; t_start = time.time()
    if device == "cuda": torch.cuda.reset_peak_memory_stats()
    print(f"[{tag}] {n_ep} cases/epoch x {epochs} epochs = {steps_per_epoch * epochs} optimizer steps of {per_step} cases | sampling {SAMPLING} | loss {LOSS}")
    for ep in range(1, epochs + 1):
        model.train(); order = sample_epoch(train_docs, n_ep, rng); run, nb = 0.0, 0; t_ep = time.time()
        for s in range(steps_per_epoch):
            f = (step + 1) / warm if step < warm else 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * f
            for a in range(GRAD_ACCUM):
                j = (s * GRAD_ACCUM + a) * DOCS_PER_MICRO; b = collate(order[j:j + DOCS_PER_MICRO])
                with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
                    logits, _ = model(b)
                loss = multilabel_loss(logits, b["Y"], pw)
                if use_kd: loss = (1 - EXP["kd_alpha"]) * loss + EXP["kd_alpha"] * kd_loss(logits, b["teach"], EXP["kd_temp"])
                if not math.isfinite(loss.item()): continue
                scaler.scale(loss / GRAD_ACCUM).backward(); run += loss.item(); nb += 1
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.grad is not None], CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        rec = dict(epoch=ep, train_loss=run / max(nb, 1), sec=time.time() - t_ep)
        if dev_docs is not None:
            f1, t, _, _, _, sec_eval = evaluate(model, dev_docs); rec.update(dev_macro_f1=f1, dev_thr=t, eval_sec=sec_eval)
            if f1 > best["f1"]: best = dict(f1=f1, epoch=ep, state=snapshot(model))
            print(f"[{tag}] epoch {ep}/{epochs} | loss {rec['train_loss']:.4f} | dev macro-F1 {f1:.4f} (global threshold {t:.2f}) | train {rec['sec']:.0f}s + eval {sec_eval:.0f}s")
        else:
            print(f"[{tag}] epoch {ep}/{epochs} | loss {rec['train_loss']:.4f} | {rec['sec']:.0f}s")
        hist.append(rec)
    if best["state"] is not None: model.load_state_dict(best["state"], strict=False)
    peak = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0
    return model, dict(history=hist, best_epoch=best["epoch"] or len(hist), best_dev_f1=best["f1"], train_minutes=(time.time() - t_start) / 60, peak_gpu_gb=peak, docs_per_epoch=n_ep)

### Throughput benchmark and budget fitting
Measures seconds per training case (forward + backward) and per dev case (forward only). If `EPOCHS` passes over every training case plus the dev evaluations would exceed `TARGET_TRAIN_MINUTES`, `TRAIN_DOCS_PER_EPOCH` is reduced so the whole schedule fits. Nothing is trained here.

In [11]:
use_kd = TEACHER is not None
train_pool = RAW["train"]
if use_kd:
    train_pool = [d for d in train_pool if d["key"] in TEACHER["train"]]
    assert train_pool, "the teacher file contains none of the training cases"
    for d in train_pool: d["teacher"] = TEACHER["train"][d["key"]].astype(np.float32)
    if not ALLOW_LARGER_STUDENT: assert TOTAL_P < TEACHER["meta"]["total_params"], \
        f"student ({TOTAL_P / 1e6:.1f}M) is not smaller than the teacher ({TEACHER['meta']['total_params'] / 1e6:.1f}M); pick a larger teacher or set ALLOW_LARGER_STUDENT=True"
dev_pool = RAW["dev"] if MAX_DEV_DOCS is None else RAW["dev"][:MAX_DEV_DOCS]

def benchmark(n=4):
    m = build_model(); docs = random.Random(1).sample(train_pool, min(n, len(train_pool))); dd = dev_pool[:n]; pw = class_pos_weight(train_pool)
    for d in docs + dd: get_ids(d)
    def once(fwd_only):
        t0 = time.time()
        for d in (dd if fwd_only else docs):
            b = collate([d])
            with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
                if fwd_only:
                    with torch.no_grad(): m.eval(); m(b)
                else:
                    m.train(); logits, _ = m(b); multilabel_loss(logits, b["Y"], pw).backward()
        if device == "cuda": torch.cuda.synchronize()
        return (time.time() - t0) / max(len(dd if fwd_only else docs), 1)
    once(False); once(True)
    tr, ev = once(False), once(True); m.zero_grad(set_to_none=True); del m
    return tr, ev

t_train, t_eval = benchmark()
full = len(train_pool) * EPOCHS * t_train + len(dev_pool) * EPOCHS * t_eval
print(f"benchmark: {t_train:.2f}s per training case, {t_eval:.2f}s per dev case | all {len(train_pool)} training cases x {EPOCHS} epochs + dev evals = about {full / 60:.0f} min")
if AUTO_FIT_BUDGET and TRAIN_DOCS_PER_EPOCH is None and full / 60 > TARGET_TRAIN_MINUTES:
    budget = TARGET_TRAIN_MINUTES * 60 - len(dev_pool) * EPOCHS * t_eval
    TRAIN_DOCS_PER_EPOCH = max(GRAD_ACCUM * DOCS_PER_MICRO, int(budget / (EPOCHS * t_train)))
    print(f"-> over the {TARGET_TRAIN_MINUTES}-minute budget: sampling {TRAIN_DOCS_PER_EPOCH} training cases per epoch "
          f"({TRAIN_DOCS_PER_EPOCH * EPOCHS} case-visits in total; with SAMPLING='{SAMPLING}' cases with rare statutes are drawn more often)")

benchmark: 0.10s per training case, 0.06s per dev case | all 42312 training cases x 4 epochs + dev evals = about 307 min
-> over the 90-minute budget: sampling 8246 training cases per epoch (32984 case-visits in total; with SAMPLING='rare_boost' cases with rare statutes are drawn more often)


## 9. Phase A: train, select the epoch on dev, choose thresholds

In [ ]:
model, infoA = run_training(train_pool, dev_pool, EPOCHS, f"{EXPERIMENT}/A", use_kd, TRAIN_DOCS_PER_EPOCH)
fA, tA, probsA, attA, logitA, sec_inf = evaluate(model, dev_pool); YA = gold_matrix(dev_pool)
THR, thr_info = choose_thresholds(YA, probsA, THRESHOLD_MODE); PA = apply_thr(probsA, THR); scA = ml_scores(YA, PA)
sc_g = ml_scores(YA, apply_thr(probsA, thr_info["global_thr"]))
base_P = np.zeros_like(YA); base_P[:, np.argsort(-train_freq)[:max(1, int(round(YA.sum(1).mean())))]] = True; base = ml_scores(YA, base_P)
print(f"\nbest epoch {infoA['best_epoch']} | DEV macro-F1 (statutes present in dev gold): {scA['macro_f1']:.4f} | micro-F1 {scA['micro_f1']:.4f} | subset accuracy {scA['subset_acc']:.4f}")
print(f"macro-P {scA['macro_p']:.4f} macro-R {scA['macro_r']:.4f} | macro-F1 over union {scA['macro_f1_union']:.4f}, over all {L} statutes {scA['macro_f1_all']:.4f} | frequent-statutes baseline macro-F1 {base['macro_f1']:.4f}")
print(f"statutes per case: gold {scA['avg_gold']:.2f} vs predicted {scA['avg_pred']:.2f} | dev inference {sec_inf:.0f}s for {len(dev_pool)} cases")
if scA["avg_pred"] > 3 * max(scA["avg_gold"], 1e-9) or thr_info["global_thr"] <= THR_GRID[1]:
    print(f"!! WARNING: the macro-F1-optimal threshold ({thr_info['global_thr']:.2f}) makes the model predict {scA['avg_pred']:.1f} statutes per case against {scA['avg_gold']:.1f} in the gold data. "
          "The model has not learned to separate statutes yet (too little training, a bad learning rate or a frozen backbone that is too weak); do not rely on this submission, consider more epochs, a higher TARGET_TRAIN_MINUTES or a different technique.")
print(f"thresholds: global {thr_info['global_thr']:.2f} -> macro-F1 {sc_g['macro_f1']:.4f} (in-sample) | honest 2-fold estimate: global {thr_info.get('honest_global', float('nan')):.4f} vs per-label {thr_info.get('honest_per_label', float('nan')):.4f} "
      f"-> using '{thr_info['mode']}' (in-sample {scA['macro_f1']:.4f})")

### Rare statutes, confusions and label co-occurrence
F1 by how often a statute appears in training, the statutes most often missed or wrongly predicted, which (missed -> spurious) pairs occur together in the same case, and whether predicted statute pairs also occur together in the training data.

In [ ]:
f1l = scA["per_label_f1"]; support = YA.sum(0); present = support > 0
print("macro-F1 by training frequency of the statute (dev statutes present in gold):")
buckets = [("not in train", 0, 0), ("1-5", 1, 5), ("6-20", 6, 20), ("21-100", 21, 100), (">100", 101, 10**9)]
bucket_f1 = {}
for name, lo, hi in buckets:
    m = present & (train_freq >= lo) & (train_freq <= hi)
    if m.any(): bucket_f1[name] = (float(f1l[m].mean()), int(m.sum())); print(f"   {name:13s} statutes {int(m.sum()):4d} | mean F1 {f1l[m].mean():.3f}")
tp = (YA & PA).sum(0); fp = (~YA & PA).sum(0); fn = (YA & ~PA).sum(0)
miss = np.argsort(-fn)[:8]; spur = np.argsort(-fp)[:8]
print("most often MISSED statutes (id: missed/support):", [(LABEL_IDS[j], int(fn[j]), int(support[j])) for j in miss if fn[j] > 0])
print("most often WRONGLY predicted (id: false positives):", [(LABEL_IDS[j], int(fp[j])) for j in spur if fp[j] > 0])
pairs = Counter()
for i in range(len(YA)):
    for a in np.where(YA[i] & ~PA[i])[0]:
        for b in np.where(~YA[i] & PA[i])[0]: pairs[(LABEL_IDS[a], LABEL_IDS[b])] += 1
print("most frequent confusions (missed statute -> predicted instead):", pairs.most_common(6))
co = Counter()
for d in RAW["train"]:
    for a in range(len(d["yi"])):
        for b in range(a + 1, len(d["yi"])): co[(d["yi"][a], d["yi"][b])] += 1; co[(d["yi"][b], d["yi"][a])] += 1
pred_pairs = [(a, b) for i in range(len(PA)) for a in np.where(PA[i])[0] for b in np.where(PA[i])[0] if a < b]
seen_frac = np.mean([(a, b) in co for a, b in pred_pairs]) if pred_pairs else float("nan")
gold_pairs = [(a, b) for i in range(len(YA)) for a in np.where(YA[i])[0] for b in np.where(YA[i])[0] if a < b]
print(f"co-occurrence: {seen_frac:.1%} of predicted statute pairs also co-occur in some training case (gold pairs in dev: {np.mean([(a, b) in co for a, b in gold_pairs]) if gold_pairs else float('nan'):.1%})")
print("top co-occurring statute pairs in train:", [((LABEL_IDS[a], LABEL_IDS[b]), c) for (a, b), c in co.most_common(8) if a < b][:4])

### Which parts of a case matter? (section probe and attention)
The trained model is scored on dev with different parts of the document as input (first/last chunk only, first/last/head+tail/uniform `MAX_CHUNKS`, and twice as many uniform chunks), each with its own best global threshold, so the numbers compare ranking quality. Inference only; the model was trained with `CHUNK_POLICY`, so other policies are a small distribution shift.

In [ ]:
probe_docs = dev_pool[:PROBE_MAX_DOCS]; YP = gold_matrix(probe_docs); probe = {}
for name, (pol, M) in {"first chunk only": ("first", 1), "last chunk only": ("last", 1), f"first {MAX_CHUNKS}": ("first", MAX_CHUNKS), f"last {MAX_CHUNKS}": ("last", MAX_CHUNKS),
                       f"head+tail {MAX_CHUNKS}": ("head_tail", MAX_CHUNKS), f"uniform {MAX_CHUNKS}": ("uniform", MAX_CHUNKS), f"uniform {2 * MAX_CHUNKS}": ("uniform", 2 * MAX_CHUNKS)}.items():
    pr, _, _ = predict(model, probe_docs, pol, M); probe[name] = best_global(YP, pr)[1]
print(f"dev macro-F1 on {len(probe_docs)} cases by input section (trained with '{CHUNK_POLICY}', {MAX_CHUNKS} chunks):")
for k, v in probe.items(): print(f"   {k:18s} {v:.4f}")
att_by_pos = None
if attA:
    acc = np.zeros(8); cnt = np.zeros(8)
    for d, w, yi in zip(dev_pool, attA, [d["yi"] for d in dev_pool]):
        sel, _ = select_chunks(len(get_ids(d)), CHUNK_POLICY, MAX_CHUNKS); wm = w[:, yi].mean(1) if w.ndim == 2 and w.shape[1] == L else w
        for (st, en, rel), wi in zip(sel, wm): bkt = min(int(rel * 8), 7); acc[bkt] += wi; cnt[bkt] += 1
    att_by_pos = (acc / np.maximum(cnt, 1)).tolist()
    print("mean attention weight (on the case's gold statutes) by relative position (octiles, 0 = start):", [f"{a:.2f}" for a in att_by_pos])

## 10. Save teacher outputs, optional final retrain, predict the test set, write and validate the submission

In [ ]:
def save_teacher(path, meta):
    out = {"meta": meta, "label_ids": LABEL_IDS, "train": {}, "dev": {}, "test": {}}
    sub = random.Random(SEED).sample(RAW["train"], min(TEACHER_SAVE_TRAIN_DOCS, len(RAW["train"])))
    for name, docs in (("train", sub), ("dev", RAW["dev"]), ("test", RAW["test"])):
        _, _, lg = predict(model, docs)
        for d, l in zip(docs, lg): out[name][d["key"]] = l.astype(np.float16)
    with open(path, "wb") as f: pickle.dump(out, f)

TEACHER_OUT = f"{EXP_DIR}/teacher_{EXPERIMENT}.pkl"
if SAVE_TEACHER and not use_kd:
    t0 = time.time(); save_teacher(TEACHER_OUT, dict(experiment=EXPERIMENT, total_params=TOTAL_P, dev_macro_f1=scA["macro_f1"], note="Phase-A model"))
    print(f"saved teacher outputs -> {TEACHER_OUT} ({time.time() - t0:.0f}s). Use this path as TEACHER_PKL for a *_kd preset.")

if FINAL_RETRAIN:
    del model
    if device == "cuda": torch.cuda.empty_cache()
    for d in RAW["dev"]:
        if use_kd: d["teacher"] = TEACHER["dev"][d["key"]].astype(np.float32)
    model, infoB = run_training(train_pool + RAW["dev"], None, max(2, infoA["best_epoch"]), f"{EXPERIMENT}/B", use_kd, TRAIN_DOCS_PER_EPOCH)
else:
    infoB = dict(note="test predictions come from the phase-A model (best dev epoch)", best_epoch=infoA["best_epoch"])

t0 = time.time(); probsT, _, _ = predict(model, RAW["test"]); infer_sec = time.time() - t0
PT = apply_thr(probsT, THR)
submission = [{"id": d["id"], "pred": [int(LABEL_IDS[j]) for j in np.where(PT[i])[0]]} for i, d in enumerate(RAW["test"])]

# ---- validation of the file the scorer will read ----
valid_ids = set(LABEL_IDS)
assert len(submission) == len(RAW["test"]) and len({str(s["id"]) for s in submission}) == len(submission), "missing or duplicate cases"
assert [s["id"] for s in submission] == [d["id"] for d in RAW["test"]], "ids differ from test.json"
for s in submission:
    assert all(type(v) is int for v in s["pred"]), "pred must contain Python integers only"
    assert len(set(s["pred"])) == len(s["pred"]), "duplicate statute ids inside a prediction"
    assert set(s["pred"]) <= valid_ids, "statute id outside the known label space"
    assert len(s["pred"]) >= MIN_PRED, "empty prediction"
npred = np.array([len(s["pred"]) for s in submission])
print(f"test: {len(submission)} cases | statutes per case: mean {npred.mean():.2f}, max {npred.max()} (dev gold mean {scA['avg_gold']:.2f}) | thresholds '{thr_info['mode']}' | inference {infer_sec:.0f}s")
print("sample:", submission[:3])
PRED_PATH = f"{EXP_DIR}/prediction.json"; ZIP_PATH = f"{EXP_DIR}/submission_{EXPERIMENT}.zip"
json.dump(submission, open(PRED_PATH, "w"))
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z: z.write(PRED_PATH, arcname="prediction.json")
with zipfile.ZipFile(ZIP_PATH) as z: print("submission archive:", ZIP_PATH, "->", z.namelist())

## 11. Save the experiment record and compare runs

In [ ]:
result = dict(experiment=EXPERIMENT, description=EXP["desc"], method=EXP["method"], total_params=TOTAL_P, trainable_params_at_start=TRAIN_P,
              trainable_final=count_params(model)[1], backbone_layers=len(layers(model.backbone)), frozen_quantised_mb=FROZEN_MB, n_statutes=L,
              phaseA=dict(best_epoch=infoA["best_epoch"], dev_macro_f1=scA["macro_f1"], dev_micro_f1=scA["micro_f1"], dev_subset_acc=scA["subset_acc"],
                          dev_macro_p=scA["macro_p"], dev_macro_r=scA["macro_r"], dev_macro_f1_union=scA["macro_f1_union"], dev_macro_f1_all=scA["macro_f1_all"],
                          baseline_macro_f1=base["macro_f1"], thresholds=dict(thr_info), train_minutes=infoA["train_minutes"], peak_gpu_gb=infoA["peak_gpu_gb"],
                          docs_per_epoch=infoA["docs_per_epoch"], history=infoA["history"], section_probe=probe, attention_by_position=att_by_pos,
                          f1_by_train_frequency=bucket_f1, cooccurrence_seen_fraction=float(seen_frac), avg_pred=scA["avg_pred"], avg_gold=scA["avg_gold"]),
              phaseB=dict(infoB, test_inference_sec=infer_sec),
              config=dict(chunk_tok=CHUNK_TOK, max_chunks=MAX_CHUNKS, policy=CHUNK_POLICY, agg=AGG, loss=LOSS, sampling=SAMPLING, epochs=EPOCHS,
                          threshold_mode=thr_info["mode"], lr_backbone=EXP["lr_backbone"], lr_head=EXP["lr_head"], kd=(EXP.get("kd_alpha"), EXP.get("kd_temp"))))
json.dump(result, open(f"{OUTPUT_DIR}/results_{EXPERIMENT}.json", "w"), indent=1, default=float)
rows = []
for p in sorted(glob.glob(f"{OUTPUT_DIR}/results_*.json")):
    r = json.load(open(p)); a = r["phaseA"]
    rows.append((r["experiment"], r["method"], r["total_params"] / 1e6, r["trainable_final"] / 1e6, a["dev_macro_f1"], a["dev_micro_f1"], a["dev_subset_acc"], a["train_minutes"], a["peak_gpu_gb"]))
print("\nexperiment  | method  | total M | trainable M (end) | dev macro-F1 | micro-F1 | subset acc | train min | peak GB")
for r in rows: print(f"{r[0]:11s} | {r[1]:7s} | {r[2]:7.1f} | {r[3]:17.2f} | {r[4]:12.4f} | {r[5]:8.4f} | {r[6]:10.4f} | {r[7]:9.1f} | {r[8]:.1f}")
print("\nUpload", ZIP_PATH, "to Codabench (Phase 5), renamed to submission.zip.")

## Notes for the report and next steps
* All presets share the document representation, head, loss, sampling, thresholds and budget; differences come from the adaptation technique alone. Trainable parameters, training minutes, peak GPU memory, F1 by statute frequency, the section probe and the threshold comparison are saved in `results_*.json`.
* Ablations that need no new technique (they do **not** count toward the six): `LOSS` (`bce`/`bce_pw`/`focal`/`asl`), `AGG` (`label_attn`/`attn`/`max`/`mean`), `SAMPLING`, `THRESHOLD_MODE`, `CHUNK_POLICY`/`MAX_CHUNKS`, `LABEL_REFINE`.
* **Compute honesty:** if the benchmark cell sampled cases per epoch, the models saw only part of the training data per epoch; report this next to the scores and keep the same budget for every preset.
* The six submissions must involve different Phase-1 models and include **LORMA** and **knowledge distillation**. AdaLoRA is not implemented. The macro-F1 convention of the official scorer is unknown (the evaluator derives the label dimension from the reference data), so several variants are printed.
* Per-label thresholds are tuned on the same dev split that selects the epoch; the 2-fold estimate is the fairer number, and the Codabench score is the honest one.